In [15]:
import pandas as pd
import networkx as nx

def build_network_by_index(
    B_nodes_csv,
    AA_edges_csv,
    AB_edges_csv,
    has_header=True
):
    header = 0 if has_header else None
    G = nx.Graph()

    # Add Nodes
    nodes = pd.read_csv(B_nodes_csv, header=header)
    for i, row in nodes.iterrows():
        G.add_node(row.iloc[0], direction=row.iloc[1])

    # Add Eeges
    for edge_csv in [AA_edges_csv, AB_edges_csv]:
        edges = pd.read_csv(edge_csv, header=header)
        for i, row in edges.iterrows():
            G.add_edge(row.iloc[0],row.iloc[1],direction=row.iloc[2])

    # Define layer 0
    layer_0 = []
    for n in nodes.iloc[:, 0]:
        if G.degree(n) > 0:
            layer_0.append(n)
    
    return G,layer_0


In [17]:
def build_layers(G, layer_0, max_depth=None):
    # list of sets --> layers[d] will contain all nodes at distance d (starting will change later)
    layers = []

    
    layers.append(set(layer_0))    # "layer 0" is the source nodes
    visited = set(layer_0)    # "visited" is all nodes we have already visited
    current_layer = set(layer_0)    # nodes at the current distance from layer 0
    depth = 0    # current distance from layer 0

    
    while current_layer: #until no new nodes are found
        
        if max_depth is not None and depth >= max_depth:    # until max depth (DO NOT REMOVE KEPT FOR DEBUGGING)
            break

        next_layer = set()  # next_layer nodes = distance (depth + 1)

        for node in current_layer:    # for every node in the current layer
            for neighbor in G.neighbors(node):     # look at all neighbors of this node
                if neighbor not in visited:     # add neighbor only if it hasn't appeared in any earlier layer
                    next_layer.add(neighbor)

        if not next_layer:     # if no new nodes were found, we are done
            break

        layers.append(next_layer)  # Save next layer

        visited.update(next_layer)  # Updated all visited layers (already discovered and assigned a layer)

        current_layer = next_layer  # Current layer is the next layer
        depth = depth + 1  # Increment depth

    return layers  # return list of layers: layers[d]


In [19]:
# For each node in layer 1(d) count all edges between layer 1(d) and layer 0(d-1) --> (where d represents layer)
# For each node in layer 1(d), create two variables called pos_votes and neg_votes. 

# deletes edges based on previous layer
def filter_level(G, curr_layer, prev_layer, first):
    temp = 0 # DELETE LATER-----------------------
    for v in list(curr_layer):  # for every node v in the current layer
        
        pos_votes = 0
        neg_votes = 0
        pos_edges = []   # edges that voted positive
        neg_edges = []   # edges that voted negative

        for u in list(G.neighbors(v)):  # u is neighbors of v from previous layer
            if u in prev_layer:

                node_dir = float(G.nodes[u]["direction"])
                edge_dir = float(G[u][v]["direction"])
                vote = node_dir * edge_dir

                if vote > 0:
                    pos_votes += 1
                    pos_edges.append((u, v))
                else:
                    neg_votes += 1
                    neg_edges.append((u, v))

        # if no votes, skip (otherwise dividing by zero)
        total = pos_votes + neg_votes
        if total == 0:
            continue

        # set direction of v based on majority vote
        if pos_votes > neg_votes:
            G.nodes[v]["direction"] = 1
            minority_edges = neg_edges
        else:
            G.nodes[v]["direction"] = -1
            minority_edges = pos_edges

        # calculate puc_value --> minority / total
        G.nodes[v]["puc_value"] = len(minority_edges) / total

        if G.nodes[v]["puc_value"] > 0.2:
            # delete all edges between v and nodes in the previous layer
            if first==True: #if layer 1, remove edge
                for u in list(G.neighbors(v)):
                    if u in prev_layer:
                        G.remove_edge(u, v)
                        print(first)
            else: #all other layers, remove node
                if G.has_node(v):
                    print(first)
                    G.remove_node(v)
                    
        else:
            # delete only minority-vote edges
            for a, b in minority_edges:
                G.remove_edge(a, b)
    return G


# deletes edges in same level
def same_level_edges(G, curr_layer):
    curr_layer = set(curr_layer)  

    for u in list(curr_layer): # only consider edges where BOTH endpoints are in the same layer
        if not G.has_node(u):
            continue

        for v in list(G.neighbors(u)):
            if v in curr_layer and G.has_node(v):
                node_prod = float(G.nodes[u]["direction"]) * float(G.nodes[v]["direction"])
                edge_dir = float(G[u][v]["direction"])

                if node_prod != edge_dir:  # if product of node directions != edge direction, delete edge
                    if G.has_edge(u, v):  # avoid double-delete since undirected
                        G.remove_edge(u, v)

    return G


In [21]:
import pandas as pd

#puc_delete_threshold=0.2
graph_A, layer_0 = build_network_by_index(
    "1. Filtered Files/cpx_nodes.csv",
    "1. Filtered Files/pls-pls.csv",
    "1. Filtered Files/pls-cpx.csv",
    has_header=True
)


layers = build_layers(graph_A, layer_0, max_depth=None)
print(graph_A)

d = 1 #Start from distance = 1
while d < len(layers):
    
    # ------------------------------------------------------------------------------------------------------------
    print("Total layers:", len(layers))
    # Number of nodes per layer
    sum = 0
    for i, layer in enumerate(layers):
        if i>0:
            sum = sum + len(layer)
            print(f"Layer {i}: {len(layer)} nodes")
    print("Total number of nodes kept",sum)
    # ------------------------------------------------------------------------------------------------------------

    print("Starting with:",d)
    prev_layer = set(layers[d - 1])   # define previous layer
    curr_layer = layers[d]   # define current layer
    
    if (d == 1):
        graph_A = filter_level(graph_A, curr_layer, prev_layer, True)
    else:
        graph_A = filter_level(graph_A, curr_layer, prev_layer, False)
        
    graph_A = same_level_edges(graph_A, curr_layer)
    layers = build_layers(graph_A, layer_0, max_depth=None)
    print("done with layer:",d,"\n")
    d += 1



rows = []
for d in range(1, len(layers)):
    for node in layers[d]:
        rows.append({
            "node": node,
            "direction": graph_A.nodes[node].get("direction"),
            "puc_value": graph_A.nodes[node].get("puc_value")
        })
print(len(rows))
df = pd.DataFrame(rows, columns=["node", "direction", "puc_value"])
df.to_csv("new.csv", index=False)


Graph with 1329 nodes and 4233 edges
Total layers: 4
Layer 1: 311 nodes
Layer 2: 195 nodes
Layer 3: 20 nodes
Total number of nodes kept 526
Starting with: 1
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
Tru